In [0]:
%python
dbutils.widgets.dropdown(
    "profile_type", "SELECT", ["SELECT", "SOURCE", "GENERATED"], "Profiling input"
)
dbutils.widgets.text("generation_id", "", "Expected generation ID")
selected_profile_type = dbutils.widgets.get("profile_type")
selected_generation_id = dbutils.widgets.get("generation_id").strip()

Choose SOURCE to load the development Parquet inputs or GENERATED to profile the completed outputs of Notebook 03. Run all cells after choosing the mode.

In [0]:
%python
# Pin the widget choice for this run before loading any data.
profile_type = selected_profile_type
expected_generation_id = selected_generation_id
if profile_type not in ("SOURCE", "GENERATED"):
    raise ValueError("Set the profile_type widget to SOURCE or GENERATED and run the notebook from the top.")

source_aliases = {
    "PERSON": "bronze_cis_mdm.TOS_PERSON",
    "CUSTOMER_ACCOUNT_REL": "bronze_cis_mdm.TOS_CUSTOMER_ACCOUNT_RELATIONSHIP",
    "CUSTOMER_RELATIONSHIP": "bronze_cis_mdm.TOS_CUSTOMER_RELATIONSHIP",
    "TKQ_ACCOUNT": "bronze_tsys.KQ_TKQ_ACCOUNT",
    "TKQ_ACCOUNT_BALANCE": "bronze_tsys.KQ_TKQ_ACCOUNT_BALANCE",
    "TRADELINES": "bronze_transunion.tradlines_file",
}

if profile_type == "SOURCE":
    for logical_name in source_aliases:
        path = f"/Volumes/workspace/default/scotia/RCAR_synthetic_{logical_name}.parquet"
        spark.read.parquet(path).write.format("delta").mode("overwrite").option(
            "overwriteSchema", "true"
        ).saveAsTable(f"workspace.default.{logical_name}")

In [0]:
%python
if profile_type == "SOURCE":
    for schema in ("bronze_cis_mdm", "bronze_tsys", "bronze_transunion"):
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.{schema}")

    for logical_name, alias in source_aliases.items():
        spark.sql(
            f"CREATE OR REPLACE VIEW workspace.{alias} AS "
            f"SELECT * FROM workspace.default.{logical_name}"
        )

In [0]:
-- Profiling metadata
USE CATALOG workspace;
CREATE SCHEMA IF NOT EXISTS synthetic_profile;

In [0]:
%python
from pyspark.sql import functions as F

if profile_type == "GENERATED":
    import sys
    from pathlib import Path

    repo_dirs = (Path.cwd(), Path.cwd().parent, *(Path(path) for path in sys.path if path))
    for repo_dir in repo_dirs:
        src_dir = repo_dir if repo_dir.name == "src" else repo_dir / "src"
        if (src_dir / "synthetic_banking" / "validation.py").is_file():
            if str(src_dir) not in sys.path:
                sys.path.insert(0, str(src_dir))
            break
    else:
        raise RuntimeError("Open Notebook 01 from a Databricks Git folder containing this repository and src/.")

    from synthetic_banking.validation import check_validated_versions


generation_state_table = "workspace.synthetic_generated.generation_state"


def completed_generation():
    if not spark.catalog.tableExists(generation_state_table):
        raise RuntimeError("Run Notebook 03 through the Delta save cell before profiling GENERATED data.")

    state = spark.table(generation_state_table).take(2)
    if (
        len(state) != 1
        or state[0]["status"] != "COMPLETED"
        or not state[0]["generation_id"]
        or not state[0]["source_run_id"]
    ):
        raise RuntimeError("There is no completed generation batch. Run Notebook 03 through the Delta save cell.")
    check_validated_versions(spark, state[0].asDict().get("validated_versions"))
    return state[0]


generation_id = None
baseline_source_run_id = None
if profile_type == "SOURCE":
    input_objects = source_aliases
else:
    generation = completed_generation()
    generation_id = generation["generation_id"]
    if expected_generation_id and generation_id != expected_generation_id:
        raise RuntimeError(f"Expected generation {expected_generation_id}, but the current batch is {generation_id}.")
    baseline_source_run_id = generation["source_run_id"]
    baseline = (
        spark.table("workspace.synthetic_profile.profile_runs")
        .filter(
            (F.col("run_id") == baseline_source_run_id)
            & (F.col("profile_type") == "SOURCE")
            & (F.col("status") == "COMPLETED")
            & F.col("completed_at").isNotNull()
        )
        .first()
    )
    if baseline is None:
        raise RuntimeError(f"Generation references an unavailable SOURCE profile: {baseline_source_run_id}.")

    # Notebook 03 currently writes five tables.
    input_objects = {
        name: f"synthetic_generated.{name}"
        for name in source_aliases
        if name != "CUSTOMER_RELATIONSHIP"
    }

profile_inputs = {name: f"workspace.{obj}" for name, obj in input_objects.items()}
missing_inputs = [obj for obj in profile_inputs.values() if not spark.catalog.tableExists(obj)]
if missing_inputs:
    raise RuntimeError(f"Missing {profile_type} profiling inputs: {', '.join(missing_inputs)}")

for name, obj in profile_inputs.items():
    spark.table(obj).createOrReplaceTempView(f"profile_input_{name.lower()}")

input_config = spark.createDataFrame(
    [(name, *obj.split("."), obj) for name, obj in input_objects.items()],
    "logical_table_name STRING, source_schema STRING, source_table STRING, source_object STRING",
)
input_config.createOrReplaceTempView("active_profile_inputs")
if profile_type == "SOURCE":
    input_config.write.format("delta").mode("overwrite").saveAsTable(
        "workspace.synthetic_profile.source_config"
    )

print(f"{profile_type} inputs: {', '.join(profile_inputs.values())}")


def create_profile_feature(name, query):
    # The source view definitions use only fully qualified, allowlisted objects.
    query = query.format(**profile_inputs)
    spark.sql(query).createOrReplaceTempView(f"active_{name}")
    if profile_type == "SOURCE":
        spark.sql(f"CREATE OR REPLACE VIEW workspace.synthetic_profile.{name} AS {query}")

Register a profiling run after validating its inputs.

In [0]:
%python
from datetime import datetime, timezone
from uuid import uuid4

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.synthetic_profile.profile_runs (
    run_id STRING,
    started_at TIMESTAMP,
    completed_at TIMESTAMP,
    status STRING,
    profile_type STRING
) USING DELTA
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.synthetic_profile.current_run (
    run_id STRING,
    started_at TIMESTAMP
) USING DELTA
""")

run_id = str(uuid4())
spark.createDataFrame(
    [(run_id, datetime.now(timezone.utc), profile_type, baseline_source_run_id, generation_id)],
    "run_id STRING, started_at TIMESTAMP, profile_type STRING, source_run_id STRING, generation_id STRING",
).createOrReplaceTempView("active_profile_run")

spark.sql("""
INSERT INTO workspace.synthetic_profile.profile_runs
    (run_id, started_at, completed_at, status, profile_type)
SELECT run_id, started_at, CAST(NULL AS TIMESTAMP), 'RUNNING', profile_type
FROM active_profile_run
""")
print(f"Started {profile_type} profile {run_id}.")
if generation_id:
    print(f"Generation {generation_id} will be profiled against SOURCE run {baseline_source_run_id}.")

In [0]:
CREATE OR REPLACE TEMP VIEW active_column_schema AS

SELECT
    s.logical_table_name,
    s.source_schema,
    s.source_table,
    c.column_name,
    c.ordinal_position,
    c.data_type,
    c.is_nullable

FROM active_profile_inputs s

JOIN workspace.information_schema.columns c
    ON lower(c.table_schema) = lower(s.source_schema)
   AND lower(c.table_name)   = lower(s.source_table);

In [0]:
SELECT *
FROM active_column_schema
ORDER BY logical_table_name, ordinal_position;

In [0]:
%python
inventory_df = spark.sql("""
SELECT
    *,
    CASE

        WHEN upper(column_name) IN (
            'PARTY_ID',
            'FROM_PARTY_ID',
            'TO_PARTY_ID',
            'ACCT_ID',
            'ACCOUNT_ID',
            'ACCOUNT_NUMBER',
            'REQ-CIF-KEY',
            'REQ-CIF-TIE-BREAKER'
        )
        THEN 'IDENTIFIER'

        WHEN upper(data_type) IN (
            'BYTE', 'SHORT', 'INT', 'INTEGER',
            'LONG', 'BIGINT',
            'FLOAT', 'DOUBLE',
            'DECIMAL'
        )
        THEN 'NUMERIC'

        WHEN upper(data_type) IN ('DATE')
        THEN 'DATE'

        WHEN upper(data_type) IN (
            'TIMESTAMP',
            'TIMESTAMP_NTZ'
        )
        THEN 'TIMESTAMP'

        WHEN upper(data_type) IN ('STRING', 'VARCHAR', 'CHAR')
        THEN 'CATEGORICAL_CANDIDATE'

        ELSE 'OTHER'

    END AS column_role
FROM active_column_schema
""")
inventory_df.createOrReplaceTempView("active_column_inventory")
if profile_type == "SOURCE":
    inventory_df.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable("workspace.synthetic_profile.column_inventory")

In [0]:
USE CATALOG workspace;

CREATE SCHEMA IF NOT EXISTS synthetic_profile;

In [0]:
%sql

CREATE TABLE IF NOT EXISTS workspace.synthetic_profile.column_quality (
    run_id STRING,
    logical_table_name STRING,
    column_name STRING,
    data_type STRING,
    row_count BIGINT,
    non_null_count BIGINT,
    null_count BIGINT,
    null_pct DOUBLE,
    distinct_count BIGINT,
    distinct_pct DOUBLE
)
USING DELTA;


In [0]:
%sql

DELETE FROM workspace.synthetic_profile.column_quality
WHERE run_id = (
    SELECT run_id
    FROM active_profile_run
);

In [0]:
%python
from collections import defaultdict
from pyspark.sql import functions as F


# Get current profiling run


run_id = (
    spark.table("active_profile_run")
    .select("run_id")
    .first()["run_id"]
)


# Read column inventory


inventory = (
    spark.table("active_column_inventory")
    .select(
        "logical_table_name",
        "source_schema",
        "source_table",
        "column_name",
        "data_type"
    )
    .collect()
)

# Group columns by source table
tables = defaultdict(list)

for row in inventory:
    key = (
        row["logical_table_name"],
        row["source_schema"],
        row["source_table"]
    )

    tables[key].append(
        (
            row["column_name"],
            row["data_type"]
        )
    )



# Profile one table at a time


profile_rows = []

for (
    logical_table_name,
    source_schema,
    source_table
), columns in tables.items():

    full_table_name = (
        f"workspace.{source_schema}.{source_table}"
    )

    print(f"Profiling {full_table_name}")

    df = spark.table(full_table_name)

    # Total number of rows once per table
    row_count = df.count()

    aggregation_expressions = []

    for column_name, data_type in columns:

        c = F.col(column_name)

        aggregation_expressions.extend([
            F.count(c).alias(
                f"{column_name}__non_null"
            ),

            # Much faster than exact COUNT(DISTINCT)
            F.approx_count_distinct(c).alias(
                f"{column_name}__distinct"
            )
        ])

    # ONE aggregation query for the entire table
    stats = df.agg(
        *aggregation_expressions
    ).first()

    # Convert wide aggregation results into profiling rows
    for column_name, data_type in columns:

        non_null_count = stats[
            f"{column_name}__non_null"
        ]

        distinct_count = stats[
            f"{column_name}__distinct"
        ]

        null_count = row_count - non_null_count

        null_pct = (
            (null_count / row_count) * 100
            if row_count > 0
            else None
        )

        distinct_pct = (
            (distinct_count / row_count) * 100
            if row_count > 0
            else None
        )

        profile_rows.append({
            "run_id": run_id,
            "logical_table_name": logical_table_name,
            "column_name": column_name,
            "data_type": data_type,
            "row_count": row_count,
            "non_null_count": non_null_count,
            "null_count": null_count,
            "null_pct": null_pct,
            "distinct_count": distinct_count,
            "distinct_pct": distinct_pct
        })



# Save profiling results


profile_df = spark.createDataFrame(profile_rows)

profile_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "workspace.synthetic_profile.column_quality"
    )

In [0]:
%sql

SELECT *
FROM workspace.synthetic_profile.column_quality
ORDER BY
    logical_table_name,
    column_name;

Blank counts

In [0]:
%sql

SELECT
    PARTY_ID,
    COUNT(*) AS record_count
FROM profile_input_person
GROUP BY PARTY_ID
HAVING COUNT(*) > 1;

In [0]:
%sql

SELECT
    ACCOUNT_ID,
    COUNT(*) AS record_count
FROM profile_input_tkq_account
GROUP BY ACCOUNT_ID
HAVING COUNT(*) > 1;

In [0]:
%sql

SELECT
    ACCOUNT_ID,
    BUSINESSEFFECTIVEDATE,
    COUNT(*) AS record_count
FROM profile_input_tkq_account
GROUP BY
    ACCOUNT_ID,
    BUSINESSEFFECTIVEDATE
HAVING COUNT(*) > 1;

In [0]:
SELECT
    COUNT(*) AS missing_person_relationships

FROM profile_input_customer_account_rel car

LEFT JOIN profile_input_person p
    ON car.PARTY_ID = p.PARTY_ID

WHERE p.PARTY_ID IS NULL;

SELECT
    COUNT(*) AS missing_accounts

FROM profile_input_customer_account_rel car

LEFT JOIN (
    SELECT DISTINCT ACCOUNT_ID
    FROM profile_input_tkq_account
) a
    ON car.ACCT_ID = a.ACCOUNT_ID

WHERE a.ACCOUNT_ID IS NULL;

SELECT
    COUNT(*) AS balance_without_account

FROM (
    SELECT DISTINCT ACCOUNT_ID
    FROM profile_input_tkq_account_balance
) b

LEFT JOIN (
    SELECT DISTINCT ACCOUNT_ID
    FROM profile_input_tkq_account
) a
    ON b.ACCOUNT_ID = a.ACCOUNT_ID

WHERE a.ACCOUNT_ID IS NULL;

Extract statistical information

In [0]:
CREATE TABLE IF NOT EXISTS synthetic_profile.numeric_profile (
    run_id              STRING,
    logical_table_name  STRING,
    column_name         STRING,

    non_null_count      BIGINT,

    min_value           DOUBLE,
    max_value           DOUBLE,
    mean_value          DOUBLE,
    stddev_value        DOUBLE,

    p01                 DOUBLE,
    p05                 DOUBLE,
    p25                 DOUBLE,
    p50                 DOUBLE,
    p75                 DOUBLE,
    p95                 DOUBLE,
    p99                 DOUBLE
);

In [0]:
%python
from collections import defaultdict
from pyspark.sql import functions as F


# NUMERIC PROFILING
# Profiles all numeric columns table-by-table


# Current profiling run
run_id = (
    spark.table("active_profile_run")
    .select("run_id")
    .first()["run_id"]
)

# Pull only numeric columns from the inventory
inventory = (
    spark.table("active_column_inventory")
    .filter(F.col("column_role") == "NUMERIC")
    .select(
        "logical_table_name",
        "source_schema",
        "source_table",
        "column_name"
    )
    .collect()
)

# Group numeric columns by source table
tables = defaultdict(list)

for row in inventory:
    key = (
        row["logical_table_name"],
        row["source_schema"],
        row["source_table"]
    )

    tables[key].append(row["column_name"])


# Final rows to write to numeric_profile
profile_rows = []

percentile_levels = [
    0.01,
    0.05,
    0.25,
    0.50,
    0.75,
    0.95,
    0.99
]


for (
    logical_table_name,
    source_schema,
    source_table
), columns in tables.items():

    full_table_name = (
        f"workspace.{source_schema}.{source_table}"
    )

    print(
        f"Profiling {logical_table_name}: "
        f"{len(columns)} numeric columns"
    )

    df = spark.table(full_table_name)

    agg_expressions = []

    
    # Build ALL numeric statistics for this table together
    

    for i, column_name in enumerate(columns):

        # Escape backticks for unusual column names
        escaped_column = column_name.replace("`", "``")

        numeric_col = F.col(
            f"`{escaped_column}`"
        ).cast("double")

        agg_expressions.extend([

            F.count(numeric_col)
                .alias(f"c{i}_count"),

            F.min(numeric_col)
                .alias(f"c{i}_min"),

            F.max(numeric_col)
                .alias(f"c{i}_max"),

            F.avg(numeric_col)
                .alias(f"c{i}_mean"),

            F.stddev(numeric_col)
                .alias(f"c{i}_stddev"),

            F.expr(
                f"""
                percentile_approx(
                    CAST(`{escaped_column}` AS DOUBLE),
                    array(
                        0.01,
                        0.05,
                        0.25,
                        0.50,
                        0.75,
                        0.95,
                        0.99
                    ),
                    10000
                )
                """
            ).alias(f"c{i}_percentiles")
        ])

    # ONE aggregation job for the entire table
    stats = df.agg(*agg_expressions).first()

   
    # Convert wide Spark result into one row per variable
    

    for i, column_name in enumerate(columns):

        percentiles = stats[
            f"c{i}_percentiles"
        ]

        profile_rows.append({
            "run_id": run_id,
            "logical_table_name": logical_table_name,
            "column_name": column_name,

            "non_null_count": stats[
                f"c{i}_count"
            ],

            "min_value": stats[
                f"c{i}_min"
            ],

            "max_value": stats[
                f"c{i}_max"
            ],

            "mean_value": stats[
                f"c{i}_mean"
            ],

            "stddev_value": stats[
                f"c{i}_stddev"
            ],

            "p01": (
                percentiles[0]
                if percentiles is not None
                else None
            ),

            "p05": (
                percentiles[1]
                if percentiles is not None
                else None
            ),

            "p25": (
                percentiles[2]
                if percentiles is not None
                else None
            ),

            "p50": (
                percentiles[3]
                if percentiles is not None
                else None
            ),

            "p75": (
                percentiles[4]
                if percentiles is not None
                else None
            ),

            "p95": (
                percentiles[5]
                if percentiles is not None
                else None
            ),

            "p99": (
                percentiles[6]
                if percentiles is not None
                else None
            )
        })



# Save results to Delta table


numeric_profile_df = spark.createDataFrame(
    profile_rows
)

numeric_profile_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "workspace.synthetic_profile.numeric_profile"
    )

print(
    f"Finished profiling "
    f"{len(profile_rows)} numeric variables."
)

In [0]:
CREATE TABLE IF NOT EXISTS synthetic_profile.category_profile (
    run_id              STRING,
    logical_table_name  STRING,
    column_name         STRING,
    category_value      STRING,
    record_count        BIGINT,
    proportion          DOUBLE
);

In [0]:
%python
from collections import defaultdict
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# CATEGORICAL PROFILING
# Profiles all low-cardinality categorical columns
# table-by-table


# Current profiling run
run_id = (
    spark.table("active_profile_run")
    .select("run_id")
    .first()["run_id"]
)

# ------------------------------------------------------------
# Find eligible categorical variables
# ------------------------------------------------------------

inventory = (
    spark.table("active_column_inventory")
    .alias("i")
)

quality = (
    spark.table("workspace.synthetic_profile.column_quality")
    .filter(F.col("run_id") == run_id)
    .alias("q")
)

eligible = (
    inventory
    .join(
        quality,
        (
            F.col("i.logical_table_name")
            == F.col("q.logical_table_name")
        )
        &
        (
            F.col("i.column_name")
            == F.col("q.column_name")
        ),
        "inner"
    )
    .filter(
        (F.col("i.column_role") == "CATEGORICAL_CANDIDATE")
        &
        (F.col("q.distinct_count") <= 100)
    )
    .select(
        F.col("i.logical_table_name"),
        F.col("i.source_schema"),
        F.col("i.source_table"),
        F.col("i.column_name")
    )
    .collect()
)


# Group columns by source table


tables = defaultdict(list)

for row in eligible:

    key = (
        row["logical_table_name"],
        row["source_schema"],
        row["source_table"]
    )

    tables[key].append(
        row["column_name"]
    )


# Profile one table at a time


for (
    logical_table_name,
    source_schema,
    source_table
), columns in tables.items():

    if not columns:
        continue

    full_table_name = (
        f"workspace.{source_schema}.{source_table}"
    )

    print(
        f"Profiling {logical_table_name}: "
        f"{len(columns)} categorical columns"
    )

    df = spark.table(full_table_name)

    # --------------------------------------------------------
    # Convert wide categorical columns into:
    #
    # column_name | category_value
    # ------------|---------------
    # GENDER_CD   | M
    # GENDER_CD   | F
    # EMP_TYPE_CD | FT
    # ...
    #
    # This allows ONE GROUP BY for all categorical columns
    # in this table.
    # --------------------------------------------------------

    category_structs = []

    for column_name in columns:

        escaped_column = (
            column_name.replace("`", "``")
        )

        category_structs.append(
            F.struct(
                F.lit(column_name).alias(
                    "column_name"
                ),
                F.col(
                    f"`{escaped_column}`"
                ).cast("string").alias(
                    "category_value"
                )
            )
        )

    long_df = (
        df
        .select(
            F.explode(
                F.array(*category_structs)
            ).alias("category")
        )
        .select(
            F.col(
                "category.column_name"
            ).alias("column_name"),

            F.col(
                "category.category_value"
            ).alias("category_value")
        )
    )

    # Count each category
    counts = (
        long_df
        .groupBy(
            "column_name",
            "category_value"
        )
        .agg(
            F.count("*").alias(
                "record_count"
            )
        )
    )

    # Calculate proportion separately for each variable
    category_window = (
        Window.partitionBy("column_name")
    )

    results = (
        counts
        .withColumn(
            "proportion",
            F.col("record_count")
            /
            F.sum("record_count").over(
                category_window
            )
        )
        .withColumn(
            "run_id",
            F.lit(run_id)
        )
        .withColumn(
            "logical_table_name",
            F.lit(logical_table_name)
        )
        .select(
            "run_id",
            "logical_table_name",
            "column_name",
            "category_value",
            "record_count",
            "proportion"
        )
    )

    # Write this table's results
    results.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(
            "workspace.synthetic_profile.category_profile"
        )

print("Categorical profiling complete.")

In [0]:
CREATE TABLE IF NOT EXISTS synthetic_profile.date_profile (
    run_id              STRING,
    logical_table_name  STRING,
    column_name         STRING,
    non_null_count      BIGINT,
    earliest_date       DATE,
    latest_date         DATE
);

In [0]:
%python
from collections import defaultdict
from pyspark.sql import functions as F


# DATE / TIMESTAMP PROFILING


# Current run
run_id = (
    spark.table("active_profile_run")
    .select("run_id")
    .first()["run_id"]
)

# Pull date/timestamp columns
inventory = (
    spark.table("active_column_inventory")
    .filter(
        F.col("column_role").isin("DATE", "TIMESTAMP")
    )
    .select(
        "logical_table_name",
        "source_schema",
        "source_table",
        "column_name"
    )
    .collect()
)

# Group columns by source table
tables = defaultdict(list)

for row in inventory:
    key = (
        row["logical_table_name"],
        row["source_schema"],
        row["source_table"]
    )

    tables[key].append(row["column_name"])


profile_rows = []


# Profile one table at a time


for (
    logical_table_name,
    source_schema,
    source_table
), columns in tables.items():

    full_table_name = (
        f"workspace.{source_schema}.{source_table}"
    )

    print(
        f"Profiling {logical_table_name}: "
        f"{len(columns)} date/timestamp columns"
    )

    df = spark.table(full_table_name)

    agg_expressions = []

    for i, column_name in enumerate(columns):

        escaped_column = (
            column_name.replace("`", "``")
        )

        date_col = F.col(
            f"`{escaped_column}`"
        ).cast("date")

        agg_expressions.extend([
            F.count(
                F.col(f"`{escaped_column}`")
            ).alias(f"c{i}_count"),

            F.min(date_col)
                .alias(f"c{i}_min"),

            F.max(date_col)
                .alias(f"c{i}_max")
        ])

    # ONE aggregation for all date columns in the table
    stats = df.agg(
        *agg_expressions
    ).first()

    for i, column_name in enumerate(columns):

        profile_rows.append({
            "run_id": run_id,
            "logical_table_name": logical_table_name,
            "column_name": column_name,

            "non_null_count":
                stats[f"c{i}_count"],

            "earliest_date":
                stats[f"c{i}_min"],

            "latest_date":
                stats[f"c{i}_max"]
        })



# Save results


if profile_rows:

    date_profile_df = spark.createDataFrame(
        profile_rows
    )

    date_profile_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(
            "workspace.synthetic_profile.date_profile"
        )

print(
    f"Finished profiling "
    f"{len(profile_rows)} date/timestamp variables."
)

In [0]:
%python
create_profile_feature("account_features", """
SELECT
    a.ACCOUNT_ID,
    a.BUSINESSEFFECTIVEDATE,

    a.ACCOUNT_TYPE_CD,
    a.CLIENT_PRODUCT_CD,
    a.TSYS_PRODUCT_CD,

    a.OPEN_DT,
    a.CLOSE_DT,

    a.CREDIT_LIMIT_AMT,
    a.ORIGINAL_CREDIT_LIMIT_AMT,

    a.CURRENT_CREDIT_SCORE,

    a.DELINQUENCY_CNT,
    a.PAST_DUE_INDICATO_CD,

    a.MINIMUM_PAYMENT_DUE_AMT,

    b.CURRENT_BALANCE_AMT,
    b.AVAILABLE_TO_BUY_AMT,
    b.OVERLIMIT_AMT,

    b.PAYMENT_YTD_AMT,
    b.PURCHASE_YTD_AMT,


    -- Derived utilization


    CASE
        WHEN a.CREDIT_LIMIT_AMT > 0
        THEN b.CURRENT_BALANCE_AMT /
             a.CREDIT_LIMIT_AMT
    END AS UTILIZATION_RATE,

   
    -- Total amount past due
  

    COALESCE(b.AMT_PDUE_1_30, 0)
    + COALESCE(b.AMT_PDUE_31_60, 0)
    + COALESCE(b.AMT_PDUE_61_90, 0)
    + COALESCE(b.AMT_PDUE_91_120, 0)
    + COALESCE(b.AMT_PDUE_121_150, 0)
    + COALESCE(b.AMT_PDUE_151_180, 0)
    + COALESCE(b.AMT_PDUE_181_210, 0)
    + COALESCE(b.AMT_PDUE_211_240, 0)
    + COALESCE(b.AMT_PDUE_241_270, 0)
    + COALESCE(b.AMT_PDUE_271_UP, 0)

        AS TOTAL_PAST_DUE_AMT,

   
    -- Delinquency occurrences
   

    COALESCE(b.NUM_PDUE_1_30, 0)
    + COALESCE(b.NUM_PDUE_31_60, 0)
    + COALESCE(b.NUM_PDUE_61_90, 0)
    + COALESCE(b.NUM_PDUE_91_120, 0)
    + COALESCE(b.NUM_PDUE_121_150, 0)
    + COALESCE(b.NUM_PDUE_151_180, 0)
    + COALESCE(b.NUM_PDUE_181_210, 0)
    + COALESCE(b.NUM_PDUE_211_240, 0)
    + COALESCE(b.NUM_PDUE_241_270, 0)
    + COALESCE(b.NUM_PDUE_271_UP, 0)

        AS TOTAL_PAST_DUE_COUNT,

    -- Account age relative to snapshot
    datediff(
        COALESCE(a.CLOSE_DT, a.BUSINESSEFFECTIVEDATE),
        a.OPEN_DT
    ) AS ACCOUNT_AGE_DAYS,

    datediff(
        a.BUSINESSEFFECTIVEDATE,
        a.LAST_PAYMENT_DT
    ) AS DAYS_SINCE_LAST_PAYMENT,

    datediff(
        a.BUSINESSEFFECTIVEDATE,
        a.LAST_PURCHASE_DT
    ) AS DAYS_SINCE_LAST_PURCHASE

FROM {TKQ_ACCOUNT} a

LEFT JOIN {TKQ_ACCOUNT_BALANCE} b
    ON a.ACCOUNT_ID = b.ACCOUNT_ID
   AND a.BUSINESSEFFECTIVEDATE =
       b.BUSINESSEFFECTIVEDATE
""")

In [0]:
%python
create_profile_feature("customer_account_features", """
SELECT
    PARTY_ID,

    COUNT(DISTINCT ACCT_ID)
        AS ACCOUNT_COUNT,

    SUM(
        CASE
            WHEN PRIMARY_OWNERSHIP_FLAG = 'Y'
            THEN 1
            ELSE 0
        END
    ) AS PRIMARY_ACCOUNT_COUNT

FROM {CUSTOMER_ACCOUNT_REL}

GROUP BY PARTY_ID
""")

Create features

In [0]:
%python
create_profile_feature("person_features", """
SELECT
    PARTY_ID,

    CLIENT_STAT_CD,
    CUST_TYPE_CD,
    EMP_TYPE_CD,
    GENDER_CD,
    MARITAL_STAT_CD,
    OCCUP_CAT_CD,
    PREF_LANG_CD,

    BIRTH_DT,
    SINCE_DT,

    FLOOR(
        months_between(current_date(), BIRTH_DT) / 12
    ) AS AGE_YEARS,

    FLOOR(
        months_between(current_date(), SINCE_DT) / 12
    ) AS CUSTOMER_TENURE_YEARS,

    CASE
        WHEN BANKRUPTCY_DT IS NOT NULL
        THEN 1 ELSE 0
    END AS HAS_BANKRUPTCY,

    CASE
        WHEN DECEASED_DT IS NOT NULL
        THEN 1 ELSE 0
    END AS DECEASED_FLAG

FROM {PERSON}
""")

In [0]:
%python
create_profile_feature("tradeline_features", """
SELECT
    `REQ-CIF-KEY`,
    `REQ-CIF-TIE-BREAKER`,

    account_number,
    snapshot_date,

    account_type_code,

    credit_limit_amount,
    current_balance_amount,
    high_credit_amount,
    payment_amount,

    times_30_days_late_count,
    times_60_days_late_count,
    times_90_days_late_count,

    CASE
        WHEN credit_limit_amount > 0
        THEN current_balance_amount /
             credit_limit_amount
    END AS TRADELINE_UTILIZATION_RATE,

    COALESCE(times_30_days_late_count, 0)
    + COALESCE(times_60_days_late_count, 0)
    + COALESCE(times_90_days_late_count, 0)
        AS TOTAL_LATE_COUNT,

    months_between(
        snapshot_date,
        account_open_date
    ) AS TRADELINE_AGE_MONTHS

FROM {TRADELINES}
""")

In [0]:
%python
create_profile_feature("cif_tradeline_counts", """
SELECT
    `REQ-CIF-KEY`,
    `REQ-CIF-TIE-BREAKER`,

    COUNT(*) AS TRADELINE_COUNT,

    SUM(
        CASE
            WHEN account_closed_indicator = 'Y'
            THEN 1
            ELSE 0
        END
    ) AS CLOSED_TRADELINE_COUNT

FROM {TRADELINES}

GROUP BY
    `REQ-CIF-KEY`,
    `REQ-CIF-TIE-BREAKER`
""")

In [0]:
CREATE TABLE IF NOT EXISTS synthetic_profile.relationship_profile (
    run_id            STRING,
    entity_name       STRING,
    variable_1        STRING,
    variable_2        STRING,
    metric_type       STRING,
    metric_value      DOUBLE
);

In [0]:
%sql

INSERT INTO workspace.synthetic_profile.relationship_profile

SELECT
    (
        SELECT run_id
        FROM active_profile_run
        LIMIT 1
    ) AS run_id,

    'ACCOUNT' AS entity_name,

    'CREDIT_LIMIT_AMT' AS variable_1,

    'CURRENT_BALANCE_AMT' AS variable_2,

    'PEARSON_CORRELATION' AS metric_type,

    corr(
        CREDIT_LIMIT_AMT,
        CURRENT_BALANCE_AMT
    ) AS metric_value

FROM active_account_features;

In [0]:
CREATE TABLE IF NOT EXISTS synthetic_profile.conditional_profile (
    run_id             STRING,
    entity_name        STRING,
    condition_variable STRING,
    condition_bucket   STRING,
    outcome_variable   STRING,
    record_count       BIGINT,
    outcome_rate       DOUBLE
);
CREATE TABLE IF NOT EXISTS synthetic_profile.conditional_profile (
    run_id             STRING,
    entity_name        STRING,
    condition_variable STRING,
    condition_bucket   STRING,
    outcome_variable   STRING,
    record_count       BIGINT,
    outcome_rate       DOUBLE
);

In [0]:
INSERT INTO synthetic_profile.conditional_profile

SELECT
    r.run_id,

    'ACCOUNT',

    'UTILIZATION_RATE',

    CASE
        WHEN UTILIZATION_RATE IS NULL
            THEN 'NULL'

        WHEN UTILIZATION_RATE < 0.30
            THEN '0-30%'

        WHEN UTILIZATION_RATE < 0.60
            THEN '30-60%'

        WHEN UTILIZATION_RATE < 0.80
            THEN '60-80%'

        WHEN UTILIZATION_RATE <= 1.00
            THEN '80-100%'

        ELSE '100%+'
    END,

    'PAST_DUE_INDICATO_CD',

    COUNT(*),

    AVG(
        CASE
            WHEN PAST_DUE_INDICATO_CD = 'Y'
            THEN 1.0
            ELSE 0.0
        END
    )

FROM active_account_features

CROSS JOIN active_profile_run r

GROUP BY
    r.run_id,

    CASE
        WHEN UTILIZATION_RATE IS NULL THEN 'NULL'
        WHEN UTILIZATION_RATE < 0.30 THEN '0-30%'
        WHEN UTILIZATION_RATE < 0.60 THEN '30-60%'
        WHEN UTILIZATION_RATE < 0.80 THEN '60-80%'
        WHEN UTILIZATION_RATE <= 1.00 THEN '80-100%'
        ELSE '100%+'
    END;

In [0]:
CREATE TABLE IF NOT EXISTS synthetic_profile.cardinality_profile (
    run_id             STRING,
    relationship_name  STRING,
    cardinality        INT,
    entity_count       BIGINT,
    proportion         DOUBLE
);

In [0]:
INSERT INTO synthetic_profile.cardinality_profile

SELECT
    r.run_id,

    'ACCOUNTS_PER_PARTY',

    ACCOUNT_COUNT,

    COUNT(*),

    COUNT(*) * 1.0 /
        SUM(COUNT(*)) OVER ()

FROM active_customer_account_features

CROSS JOIN active_profile_run r

GROUP BY
    r.run_id,
    ACCOUNT_COUNT;

Save statistics

In [0]:
%sql

CREATE OR REPLACE VIEW workspace.synthetic_profile.v_numeric_statistics AS

SELECT
    n.logical_table_name,
    n.column_name,

    q.row_count,
    q.null_count,
    q.null_pct,
    q.distinct_count,

    n.min_value,
    n.p01,
    n.p05,
    n.p25,
    n.p50,
    n.mean_value,
    n.p75,
    n.p95,
    n.p99,
    n.max_value,
    n.stddev_value

FROM workspace.synthetic_profile.numeric_profile n

JOIN workspace.synthetic_profile.column_quality q
    ON n.run_id = q.run_id
   AND n.logical_table_name = q.logical_table_name
   AND n.column_name = q.column_name

WHERE n.run_id = (
    SELECT run_id
    FROM workspace.synthetic_profile.current_run
    LIMIT 1
);

In [0]:
%sql

CREATE OR REPLACE VIEW workspace.synthetic_profile.v_category_statistics AS

SELECT
    c.logical_table_name,
    c.column_name,
    c.category_value,
    c.record_count,
    c.proportion

FROM workspace.synthetic_profile.category_profile c

WHERE c.run_id = (
    SELECT run_id
    FROM workspace.synthetic_profile.current_run
    LIMIT 1
);

In [0]:
%sql

CREATE OR REPLACE VIEW workspace.synthetic_profile.v_structural_statistics AS

SELECT
    c.relationship_name,
    c.cardinality,
    c.entity_count,
    c.proportion

FROM workspace.synthetic_profile.cardinality_profile c

WHERE c.run_id = (
    SELECT run_id
    FROM workspace.synthetic_profile.current_run
    LIMIT 1
);

In [0]:
%sql

CREATE OR REPLACE VIEW workspace.synthetic_profile.v_dependency_statistics AS

SELECT
    rp.entity_name AS relationship_name,
    rp.variable_1,
    rp.variable_2,
    rp.metric_type,
    rp.metric_value

FROM workspace.synthetic_profile.relationship_profile rp

WHERE rp.run_id = (
    SELECT run_id
    FROM workspace.synthetic_profile.current_run
    LIMIT 1
);

In [0]:
%python
if profile_type == "GENERATED" and completed_generation()["generation_id"] != generation_id:
    raise RuntimeError("Generated outputs changed during profiling. Rerun GENERATED profiling.")

spark.sql("""
UPDATE workspace.synthetic_profile.profile_runs
SET
    completed_at = current_timestamp(),
    status = 'COMPLETED'
WHERE status = 'RUNNING'
  AND run_id = (SELECT run_id FROM active_profile_run)
""")
spark.sql("""
INSERT OVERWRITE workspace.synthetic_profile.current_run
SELECT run_id, started_at
FROM active_profile_run
""")
print(f"Completed {profile_type} profile {run_id}.")

In [0]:
%sql

SELECT *
FROM workspace.synthetic_profile.v_numeric_statistics
ORDER BY logical_table_name, column_name;

In [0]:
%sql

SELECT *
FROM workspace.synthetic_profile.v_category_statistics
ORDER BY logical_table_name, column_name, proportion DESC;

In [0]:
%sql

SELECT *
FROM workspace.synthetic_profile.v_dependency_statistics;

In [0]:
%sql

SELECT *
FROM workspace.synthetic_profile.v_structural_statistics;

In [0]:
%sql

SELECT * FROM workspace.synthetic_profile.column_quality;

In [0]:
%sql

SELECT * FROM workspace.synthetic_profile.numeric_profile;

In [0]:
%sql

SELECT * FROM workspace.synthetic_profile.category_profile;

In [0]:
%sql

SELECT * FROM workspace.synthetic_profile.category_profile;

In [0]:
%sql

SELECT * FROM workspace.synthetic_profile.date_profile;

In [0]:
%sql

SELECT *
FROM workspace.synthetic_profile.profile_runs
ORDER BY started_at DESC;

Compare this completed GENERATED profile with the SOURCE run that supplied Notebook 03.

In [0]:
SELECT
    r.profile_type,

    n.logical_table_name,
    n.column_name,

    n.mean_value,
    n.p50,
    n.p95,
    n.stddev_value

FROM workspace.synthetic_profile.numeric_profile n

JOIN workspace.synthetic_profile.profile_runs r
    ON n.run_id = r.run_id

WHERE n.column_name = 'CREDIT_LIMIT_AMT'
  AND r.status = 'COMPLETED'

ORDER BY
    r.profile_type;

In [0]:
%sql

WITH selected_runs AS (

    SELECT
        r.run_id,
        r.profile_type

    FROM workspace.synthetic_profile.profile_runs r
    CROSS JOIN active_profile_run active

    WHERE active.profile_type = 'GENERATED'
      AND r.status = 'COMPLETED'
      AND (
          (r.profile_type = 'SOURCE' AND r.run_id = active.source_run_id)
          OR (r.profile_type = 'GENERATED' AND r.run_id = active.run_id)
      )
),

stats AS (

    SELECT
        n.logical_table_name,
        n.column_name,
        r.profile_type,

        n.mean_value,
        n.p25,
        n.p50,
        n.p75,
        n.p95

    FROM workspace.synthetic_profile.numeric_profile n

    JOIN selected_runs r
        ON n.run_id = r.run_id
),

comparison AS (

    SELECT
        logical_table_name,
        column_name,

        -- Mean
        MAX(
            CASE
                WHEN profile_type = 'SOURCE'
                THEN mean_value
            END
        ) AS source_mean,

        MAX(
            CASE
                WHEN profile_type = 'GENERATED'
                THEN mean_value
            END
        ) AS generated_mean,

        -- P25
        MAX(
            CASE
                WHEN profile_type = 'SOURCE'
                THEN p25
            END
        ) AS source_p25,

        MAX(
            CASE
                WHEN profile_type = 'GENERATED'
                THEN p25
            END
        ) AS generated_p25,

        -- Median / P50
        MAX(
            CASE
                WHEN profile_type = 'SOURCE'
                THEN p50
            END
        ) AS source_p50,

        MAX(
            CASE
                WHEN profile_type = 'GENERATED'
                THEN p50
            END
        ) AS generated_p50,

        -- P75
        MAX(
            CASE
                WHEN profile_type = 'SOURCE'
                THEN p75
            END
        ) AS source_p75,

        MAX(
            CASE
                WHEN profile_type = 'GENERATED'
                THEN p75
            END
        ) AS generated_p75,

        -- P95
        MAX(
            CASE
                WHEN profile_type = 'SOURCE'
                THEN p95
            END
        ) AS source_p95,

        MAX(
            CASE
                WHEN profile_type = 'GENERATED'
                THEN p95
            END
        ) AS generated_p95

    FROM stats

    GROUP BY
        logical_table_name,
        column_name
)

SELECT
    logical_table_name,
    column_name,

    -- ========================================================
    -- MEAN
    -- ========================================================

    source_mean,
    generated_mean,

    ROUND(
        100.0 *
        (generated_mean - source_mean)
        /
        NULLIF(ABS(source_mean), 0),
        2
    ) AS mean_pct_diff,

    -- ========================================================
    -- P25
    -- ========================================================

    source_p25,
    generated_p25,

    ROUND(
        100.0 *
        (generated_p25 - source_p25)
        /
        NULLIF(ABS(source_p25), 0),
        2
    ) AS p25_pct_diff,

    -- ========================================================
    -- P50 / MEDIAN
    -- ========================================================

    source_p50,
    generated_p50,

    ROUND(
        100.0 *
        (generated_p50 - source_p50)
        /
        NULLIF(ABS(source_p50), 0),
        2
    ) AS p50_pct_diff,

    -- ========================================================
    -- P75
    -- ========================================================

    source_p75,
    generated_p75,

    ROUND(
        100.0 *
        (generated_p75 - source_p75)
        /
        NULLIF(ABS(source_p75), 0),
        2
    ) AS p75_pct_diff,

    -- ========================================================
    -- P95
    -- ========================================================

    source_p95,
    generated_p95,

    ROUND(
        100.0 *
        (generated_p95 - source_p95)
        /
        NULLIF(ABS(source_p95), 0),
        2
    ) AS p95_pct_diff

FROM comparison

ORDER BY
    logical_table_name,
    column_name;